# Futbol Veri Projesi 1: Maç Verisini Toplamak ve Temizlemek

[![Colab'da Aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haticeozbolat01/veri-projeleri/blob/main/01-futbol-mac-verisi-toplama-temizleme.ipynb)

Bu not defteri, haticeozbolat.com'daki **Futbol Veri Projesi 1** yazısının kodlarını içeriyor. Yazı: [Yazıyı oku]([YAZI LİNKİ])

**Veri kaynağı:** [openfootball/football.json](https://github.com/openfootball/football.json), CC0 1.0 (kamu malı) lisanslı, gönüllülerin derlediği açık futbol verisi. Resmî bir kaynak değil; eksik ya da hatalı kayıtlar olabilir.

Hücreleri sırayla **Shift + Enter** ile çalıştır.

## 1. Veriyi indirmek

In [ ]:
import requests

URL = "https://raw.githubusercontent.com/openfootball/football.json/master/2024-25/tr.1.json"
cevap = requests.get(URL, timeout=30)
cevap.raise_for_status()
ham = cevap.json()

print(ham["name"])
print(len(ham["matches"]), "maç")

In [ ]:
ham["matches"][0]

## 2. JSON'dan tabloya

In [ ]:
import pandas as pd
import numpy as np

df = pd.json_normalize(ham["matches"])
print(df.shape)
df.head()

## 3. Sütunları düzenlemek

In [ ]:
df = df.rename(columns={
    "round": "hafta", "team1": "ev", "team2": "deplasman",
    "score.ft": "skor", "score.ht": "ilk_yari", "status": "durum",
})

df["hafta"] = df["hafta"].str.extract(r"(\d+)").astype(int)
df["tarih"] = pd.to_datetime(df["date"] + " " + df["time"])

def ayir(skor, i):
    """[ev, deplasman] listesinden i. elemanı alır; skor yoksa NaN döndürür."""
    return skor[i] if isinstance(skor, list) else np.nan

df["ev_gol"] = df["skor"].apply(ayir, i=0)
df["dep_gol"] = df["skor"].apply(ayir, i=1)
df["iy_ev"] = df["ilk_yari"].apply(ayir, i=0)
df["iy_dep"] = df["ilk_yari"].apply(ayir, i=1)
df["hukmen"] = df["durum"].eq("awarded")

df = df[["hafta", "tarih", "ev", "deplasman", "ev_gol", "dep_gol", "iy_ev", "iy_dep", "hukmen"]]
df.head()

## 4. Eksik değerler

In [ ]:
df.isna().sum()

In [ ]:
df[df["ev_gol"].isna()]

In [ ]:
df[df["hukmen"]]

## 5. Tutarlılık kontrolleri

In [ ]:
print("Haftadaki maç sayıları:", df.groupby("hafta").size().unique())
print("Takım sayısı:", pd.concat([df["ev"], df["deplasman"]]).nunique())
print("Takım başına maç:", pd.concat([df["ev"], df["deplasman"]]).value_counts().unique())
print("Aynı ev-deplasman eşleşmesi en fazla:", df.groupby(["ev", "deplasman"]).size().max(), "kez")

## 6. Oynanan maçlar ve yeni sütunlar

In [ ]:
oynanan = df.dropna(subset=["ev_gol", "dep_gol"]).copy()
oynanan[["ev_gol", "dep_gol"]] = oynanan[["ev_gol", "dep_gol"]].astype(int)

oynanan["toplam_gol"] = oynanan["ev_gol"] + oynanan["dep_gol"]
oynanan["sonuc"] = np.select(
    [oynanan["ev_gol"] > oynanan["dep_gol"], oynanan["ev_gol"] < oynanan["dep_gol"]],
    ["Ev sahibi", "Deplasman"],
    default="Beraberlik",
)
print(len(oynanan), "maç")
oynanan.head()

## 7. Temiz veriyi kaydetmek

In [ ]:
oynanan.to_csv("super_lig_2024_25_temiz.csv", index=False)
print("Kaydedildi.")

## 8. İlk bakış

In [ ]:
sahada = oynanan[~oynanan["hukmen"]]

print(f"Sahada oynanan maç: {len(sahada)}")
print(f"Maç başına gol: {sahada['toplam_gol'].mean():.2f}")
print(f"Ev sahibi gol ortalaması: {sahada['ev_gol'].mean():.2f}")
print(f"Deplasman gol ortalaması: {sahada['dep_gol'].mean():.2f}")
print()
print(sahada["sonuc"].value_counts(normalize=True).round(3))

In [ ]:
sahada.sort_values("toplam_gol", ascending=False).head(3)[["hafta", "ev", "deplasman", "ev_gol", "dep_gol"]]

In [ ]:
import matplotlib.pyplot as plt

RENK = {"Ev sahibi": "#C2185B", "Beraberlik": "#A39FAD", "Deplasman": "#E07B2E"}
MURDUM, GRI = "#2B1036", "#6B6475"
ETIKET = {"Ev sahibi": "Ev sahibi kazandı", "Beraberlik": "Berabere", "Deplasman": "Deplasman kazandı"}

sira = ["Ev sahibi", "Beraberlik", "Deplasman"]
sayilar = sahada["sonuc"].value_counts().reindex(sira)
oran = sayilar / sayilar.sum()
gol = {"Ev sahibi": sahada["ev_gol"].mean(), "Deplasman": sahada["dep_gol"].mean()}

fig = plt.figure(figsize=(10, 5.4), facecolor="white")
fig.text(0.05, 0.92, "Ev sahibi maçların neredeyse yarısını kazandı", fontsize=17, fontweight="bold", color=MURDUM)
fig.text(0.05, 0.865, f"Süper Lig 2024-25 · sahada oynanan {len(sahada)} maç", fontsize=11, color=GRI)

# Üst panel: tek çubukta sonuç dağılımı
ax1 = fig.add_axes([0.05, 0.47, 0.90, 0.28])
sol = 0
for s in sira:
    w = oran[s]
    ax1.barh(0, w - 0.004, left=sol + 0.002, height=0.55, color=RENK[s])
    ax1.text(sol + w / 2, 0.02, f"%{w * 100:.0f}", ha="center", va="center", fontsize=22, fontweight="bold", color="white")
    ax1.text(sol + w / 2, -0.52, f"{ETIKET[s]}\n{sayilar[s]} maç", ha="center", va="center", fontsize=11, color=MURDUM)
    sol += w
ax1.set_xlim(0, 1); ax1.set_ylim(-0.85, 0.4); ax1.axis("off")
fig.text(0.05, 0.765, "Maç sonuçları", fontsize=12, fontweight="bold", color=MURDUM)

# Alt panel: maç başına gol
ax2 = fig.add_axes([0.20, 0.09, 0.72, 0.22])
fig.text(0.05, 0.36, "Maç başına atılan gol", fontsize=12, fontweight="bold", color=MURDUM)
for y, s in enumerate(["Deplasman", "Ev sahibi"]):
    ax2.barh(y, gol[s], height=0.55, color=RENK[s])
    ax2.text(gol[s] + 0.03, y, f"{gol[s]:.2f}".replace(".", ","), va="center", fontsize=13, fontweight="bold", color=MURDUM)
ax2.set_yticks([0, 1]); ax2.set_yticklabels(["Deplasman", "Ev sahibi"], fontsize=11, color=MURDUM)
ax2.set_xlim(0, max(gol.values()) * 1.18)
ax2.set_xticks([]); ax2.spines["bottom"].set_visible(False); ax2.tick_params(axis="y", length=0)
for k in ["top", "right", "left"]: ax2.spines[k].set_visible(False)

fig.text(0.95, 0.015, "Kaynak: openfootball (CC0) · haticeozbolat.com", ha="right", fontsize=8.5, color=GRI)
plt.show()

## Kendin dene

1. URL'deki `tr.1` yerine `en.1` yazarak aynı sezonun Premier Lig verisini indir ve aynı temizliği uygula.
2. İlk yarı skoru olan maçlarda, ilk yarıyı önde kapatan takım maçı kaç kez kazanmış?
3. Haftalara göre maç başına gol ortalamasını hesapla. Sezon ilerledikçe değişiyor mu?
4. Ev sahibi kazanma oranı en yüksek olan takım hangisi?

## Sıradaki bölüm

**Futbol Veri Projesi 2:** Temiz veriden puan tablosu, ev sahibi avantajı ve takım analizi. [YAZI LİNKİ]